# Instrumental Variables: Concepts and Intuition

When treatment is **endogenous** (correlated with unobserved factors that also affect the outcome), OLS is biased. An **instrumental variable** (IV) is a source of exogenous variation that shifts treatment but has no direct effect on the outcome.

This notebook builds intuition through simulation:
1. Why OLS fails when treatment is endogenous
2. How IV recovers the causal effect
3. Two-stage least squares (2SLS) step by step
4. What goes wrong with weak instruments
5. LATE: who the IV estimate applies to
6. What happens when the exclusion restriction is violated

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from numpy.linalg import inv

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (9, 5),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. The Endogeneity Problem

Suppose we want to estimate the causal effect of treatment $D$ on outcome $Y$. The true model is:

$$Y = 2 + 3D + 1.5U + \varepsilon$$

where $U$ is an unobserved confounder that affects both $D$ and $Y$. Because $D$ depends on $U$, OLS on $Y \sim D$ picks up both the causal effect of $D$ *and* the confounding from $U$.

In [ ]:
n = 5000
true_effect = 3

# Unobserved confounder
U = np.random.normal(0, 1, n)

# Treatment depends on confounder (endogenous!)
D = 0.5 * U + np.random.normal(0, 1, n)

# Outcome depends on treatment AND confounder
Y = 2 + true_effect * D + 1.5 * U + np.random.normal(0, 1, n)

# OLS: regress Y on D (ignoring U because it's unobserved)
ols = smf.ols('Y ~ D', data=pd.DataFrame({'Y': Y, 'D': D})).fit()

print(f"True causal effect of D on Y: {true_effect}")
print(f"OLS estimate:                 {ols.params['D']:.3f}")
print(f"OLS bias:                     {ols.params['D'] - true_effect:+.3f}")
print(f"\nOLS is biased UPWARD because U pushes both D and Y in the same direction.")
print(f"People with high U get more treatment AND have higher outcomes,")
print(f"making treatment look more effective than it really is.")

## 2. Enter the Instrument

An **instrument** $Z$ satisfies three conditions:

1. **Relevance**: $Z$ is correlated with $D$ (the first stage)
2. **Exclusion**: $Z$ affects $Y$ only through $D$ (no direct effect)
3. **Independence**: $Z$ is uncorrelated with confounders $U$

We add $Z$ to our simulation: $Z$ shifts $D$ but is independent of $U$ and has no direct effect on $Y$.

In [ ]:
# Instrument: independent of U, affects D, no direct effect on Y
Z = np.random.normal(0, 1, n)

# Treatment now depends on Z AND U
D_iv = 0.6 * Z + 0.5 * U + np.random.normal(0, 1, n)

# Outcome still depends on D and U (Z has NO direct effect)
Y_iv = 2 + true_effect * D_iv + 1.5 * U + np.random.normal(0, 1, n)

# Verify instrument conditions
print("Checking instrument conditions:")
print(f"  Corr(Z, D):  {np.corrcoef(Z, D_iv)[0,1]:.3f}  (should be nonzero: RELEVANCE)")
print(f"  Corr(Z, U):  {np.corrcoef(Z, U)[0,1]:.3f}  (should be ~0: INDEPENDENCE)")
print()

# Wald estimator (for binary Z, but works with continuous Z too via regression)
# IV estimate = Cov(Y, Z) / Cov(D, Z)
cov_YZ = np.cov(Y_iv, Z)[0, 1]
cov_DZ = np.cov(D_iv, Z)[0, 1]
wald = cov_YZ / cov_DZ

# Compare with biased OLS
ols_biased = smf.ols('Y ~ D', data=pd.DataFrame({'Y': Y_iv, 'D': D_iv})).fit()

print(f"True effect:    {true_effect}")
print(f"OLS estimate:   {ols_biased.params['D']:.3f}  (biased)")
print(f"IV estimate:    {wald:.3f}  (unbiased!)")
print(f"\nThe IV estimate uses only the variation in D that comes from Z,")
print(f"which is uncorrelated with U. This strips out the confounding.")

## 3. Two-Stage Least Squares (2SLS) Step by Step

The most common IV estimator. Two stages:

**Stage 1**: Regress $D$ on $Z$. Get predicted values $\hat{D}$. These contain only the "clean" variation in treatment coming from the instrument.

**Stage 2**: Regress $Y$ on $\hat{D}$. The coefficient on $\hat{D}$ is the IV estimate.

Why this works: $\hat{D}$ is a function of $Z$ only, so it is uncorrelated with $U$. By replacing $D$ with $\hat{D}$, we purge the endogenous variation.

In [ ]:
# === Stage 1: D on Z ===
W1 = np.column_stack([np.ones(n), Z])
beta1 = inv(W1.T @ W1) @ (W1.T @ D_iv)
D_hat = W1 @ beta1

print("=== Stage 1: D = a + b*Z ===")
print(f"  Intercept: {beta1[0]:.3f}")
print(f"  Coef on Z: {beta1[1]:.3f}")

# First-stage F-statistic
stage1 = smf.ols('D ~ Z', data=pd.DataFrame({'D': D_iv, 'Z': Z})).fit()
print(f"  F-statistic: {stage1.fvalue:.1f}  (rule of thumb: F > 10 means strong instrument)")
print()

# === Stage 2: Y on D_hat ===
W2 = np.column_stack([np.ones(n), D_hat])
beta2 = inv(W2.T @ W2) @ (W2.T @ Y_iv)

print("=== Stage 2: Y = a + b*D_hat ===")
print(f"  Intercept:    {beta2[0]:.3f}")
print(f"  Coef on D_hat: {beta2[1]:.3f}")
print()

print(f"True effect:     {true_effect}")
print(f"OLS (biased):    {ols_biased.params['D']:.3f}")
print(f"IV/2SLS:         {beta2[1]:.3f}")
print(f"Wald (Cov ratio):{wald:.3f}")
print(f"\n2SLS and Wald agree (as they should with one instrument and no controls).")

## 4. Weak Instruments

What happens when the instrument barely affects treatment? The first stage is weak, and the IV estimate becomes noisy, biased toward OLS, and unreliable. The standard diagnostic: if the first-stage F-statistic is below 10, worry.

In [ ]:
# Vary instrument strength and track IV estimate + F-statistic
strengths = np.arange(0.02, 1.01, 0.02)
results = []

for gamma in strengths:
    np.random.seed(42)
    U_s = np.random.normal(0, 1, n)
    Z_s = np.random.normal(0, 1, n)
    D_s = gamma * Z_s + 0.5 * U_s + np.random.normal(0, 1, n)
    Y_s = 2 + true_effect * D_s + 1.5 * U_s + np.random.normal(0, 1, n)

    # First stage F
    stage1_s = smf.ols('D ~ Z', data=pd.DataFrame({'D': D_s, 'Z': Z_s})).fit()
    F = stage1_s.fvalue

    # IV estimate
    cov_yz = np.cov(Y_s, Z_s)[0, 1]
    cov_dz = np.cov(D_s, Z_s)[0, 1]
    iv_est = cov_yz / cov_dz if abs(cov_dz) > 1e-6 else np.nan

    # OLS estimate
    ols_s = smf.ols('Y ~ D', data=pd.DataFrame({'Y': Y_s, 'D': D_s})).fit()

    results.append({'gamma': gamma, 'F': F, 'iv': iv_est, 'ols': ols_s.params['D']})

res = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: IV estimate vs instrument strength
ax = axes[0]
ax.plot(res['gamma'], res['iv'], 'o-', color='steelblue', markersize=3, label='IV estimate')
ax.axhline(y=true_effect, color='red', linestyle='--', linewidth=1.5, label=f'True effect = {true_effect}')
ax.axhline(y=res['ols'].mean(), color='gray', linestyle=':', label=f'OLS (biased) ~ {res["ols"].mean():.2f}')
ax.set_xlabel('Instrument strength (gamma)', fontsize=12)
ax.set_ylabel('Estimated treatment effect', fontsize=12)
ax.set_title('IV Estimate vs. Instrument Strength', fontsize=13)
ax.legend()
ax.set_ylim(0, 8)

# Right: F-statistic vs instrument strength
ax = axes[1]
ax.plot(res['gamma'], res['F'], 'o-', color='darkorange', markersize=3)
ax.axhline(y=10, color='red', linestyle='--', linewidth=1.5, label='F = 10 threshold')
ax.set_xlabel('Instrument strength (gamma)', fontsize=12)
ax.set_ylabel('First-stage F-statistic', fontsize=12)
ax.set_title('First-Stage F vs. Instrument Strength', fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

print("When gamma is small, the instrument barely moves treatment.")
print("The IV estimate is erratic and biased toward OLS.")
print("The F-statistic drops below 10, signaling a weak instrument problem.")

## 5. LATE: Who Does IV Apply To?

IV does not estimate the average treatment effect (ATE) for the whole population. It estimates the **Local Average Treatment Effect (LATE)**: the effect for **compliers**, people whose treatment status is changed by the instrument.

Three types of people:
- **Compliers**: take treatment when $Z=1$, don't when $Z=0$ (IV estimates their effect)
- **Always-takers**: take treatment regardless of $Z$ (IV tells us nothing about them)
- **Never-takers**: never take treatment regardless of $Z$ (IV tells us nothing about them)

In [ ]:
# Simulate compliers, always-takers, never-takers with DIFFERENT treatment effects
np.random.seed(42)
n_late = 6000

# Assign types: 50% compliers, 25% always-takers, 25% never-takers
types = np.random.choice(['complier', 'always-taker', 'never-taker'],
                         size=n_late, p=[0.5, 0.25, 0.25])

# Binary instrument
Z_late = np.random.binomial(1, 0.5, n_late)

# Treatment depends on type and instrument
D_late = np.zeros(n_late)
D_late[types == 'always-taker'] = 1  # always treated
D_late[types == 'never-taker'] = 0   # never treated
D_late[(types == 'complier') & (Z_late == 1)] = 1  # treated when Z=1
D_late[(types == 'complier') & (Z_late == 0)] = 0  # untreated when Z=0

# Different treatment effects by type
tau = np.zeros(n_late)
tau[types == 'complier'] = 3.0       # complier effect
tau[types == 'always-taker'] = 1.0   # always-taker effect (smaller)
tau[types == 'never-taker'] = 5.0    # never-taker effect (larger, but unobserved)

Y_late = 2 + tau * D_late + np.random.normal(0, 1, n_late)

# Compute IV (Wald) estimate
mean_Y_Z1 = Y_late[Z_late == 1].mean()
mean_Y_Z0 = Y_late[Z_late == 0].mean()
mean_D_Z1 = D_late[Z_late == 1].mean()
mean_D_Z0 = D_late[Z_late == 0].mean()
iv_late = (mean_Y_Z1 - mean_Y_Z0) / (mean_D_Z1 - mean_D_Z0)

# True effects
ate = (tau * D_late).sum() / D_late.sum() if D_late.sum() > 0 else 0
complier_ate = tau[types == 'complier'].mean()

print("Treatment effects by type:")
print(f"  Compliers:      tau = 3.0")
print(f"  Always-takers:  tau = 1.0")
print(f"  Never-takers:   tau = 5.0 (never observed getting treated)")
print()
print(f"IV/Wald estimate: {iv_late:.3f}")
print(f"True complier effect: {complier_ate:.1f}")
print(f"\nIV recovers the complier effect, NOT the population ATE.")
print(f"This is LATE: the Local Average Treatment Effect.")

# Visualize the three groups
fig, ax = plt.subplots(figsize=(10, 5))
for i, (t, color) in enumerate([('complier', 'steelblue'),
                                  ('always-taker', 'darkorange'),
                                  ('never-taker', 'gray')]):
    mask = types == t
    counts = [np.sum(mask & (Z_late == 0) & (D_late == 0)),
              np.sum(mask & (Z_late == 0) & (D_late == 1)),
              np.sum(mask & (Z_late == 1) & (D_late == 0)),
              np.sum(mask & (Z_late == 1) & (D_late == 1))]
    ax.bar([x + i*0.25 for x in range(4)], counts, width=0.25,
           color=color, label=t.replace('-', ' ').title(), edgecolor='black')

ax.set_xticks([0.25, 1.25, 2.25, 3.25])
ax.set_xticklabels(['Z=0, D=0', 'Z=0, D=1', 'Z=1, D=0', 'Z=1, D=1'])
ax.set_ylabel('Count')
ax.set_title('Who does what? Compliers, Always-takers, Never-takers')
ax.legend()
plt.tight_layout()
plt.show()

print("\nCompliers (blue) switch treatment status when Z changes.")
print("Always-takers (orange) are always treated regardless of Z.")
print("Never-takers (gray) are never treated regardless of Z.")

## 6. Exclusion Restriction Violation

The exclusion restriction says $Z$ affects $Y$ *only* through $D$. If $Z$ has even a small direct effect on $Y$, the IV estimate is biased. Unlike relevance (which we can test with F-statistics), the exclusion restriction is **untestable**. We can only argue for it on theoretical grounds.

In [ ]:
# What if Z has a direct effect on Y?
# Y = 2 + 3*D + delta*Z + 1.5*U + noise
# When delta != 0, the exclusion restriction is violated

np.random.seed(42)
deltas = np.arange(-1.0, 1.01, 0.05)
iv_estimates = []

for delta in deltas:
    U_ex = np.random.normal(0, 1, n)
    Z_ex = np.random.normal(0, 1, n)
    D_ex = 0.6 * Z_ex + 0.5 * U_ex + np.random.normal(0, 1, n)
    Y_ex = 2 + true_effect * D_ex + delta * Z_ex + 1.5 * U_ex + np.random.normal(0, 1, n)

    cov_yz = np.cov(Y_ex, Z_ex)[0, 1]
    cov_dz = np.cov(D_ex, Z_ex)[0, 1]
    iv_estimates.append(cov_yz / cov_dz)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(deltas, iv_estimates, 'o-', color='steelblue', markersize=4)
ax.axhline(y=true_effect, color='red', linestyle='--', linewidth=1.5, label=f'True effect = {true_effect}')
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5, label='Exclusion holds (delta = 0)')
ax.set_xlabel('Direct effect of Z on Y (delta)', fontsize=12)
ax.set_ylabel('IV estimate', fontsize=12)
ax.set_title('IV Bias When the Exclusion Restriction is Violated', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print(f"When delta = 0, IV gives {true_effect:.1f} (correct).")
print(f"When delta > 0, Z directly helps Y, making treatment look more effective.")
print(f"When delta < 0, Z directly hurts Y, making treatment look less effective.")
print(f"\nThe exclusion restriction cannot be tested statistically.")
print(f"You can only argue for it based on the research design.")

## Key Takeaways

1. **OLS is biased** when treatment is endogenous (correlated with unobserved confounders).

2. **An instrument** is a variable that shifts treatment but has no direct effect on the outcome. Three conditions: relevance, exclusion, independence.

3. **2SLS** isolates the "clean" variation in treatment (coming from the instrument) and uses only that to estimate the causal effect.

4. **Weak instruments** (F < 10) make IV estimates noisy and biased toward OLS. Always report the first-stage F.

5. **LATE**: IV estimates the effect for compliers only, not for everyone. If treatment effects vary across people, IV and ATE can differ.

6. **The exclusion restriction is untestable.** You can only argue for it based on your research design. A small violation can meaningfully bias the IV estimate.

**Real-world examples from this week's readings:**
- Djourelova (2023): AP style ban as instrument for slanted language exposure
- Mueller & Schwarz (2021): internet outages as instrument for social media exposure

## 7. Simulating Djourelova (2023): The AP Ban and Immigration Attitudes

We cannot access the original data, but we can build a simulation calibrated to the paper's reported statistics. This lets us see the shift-share IV design in action and verify that the method recovers the right answer.

**Setup from the paper:**
- ~3,100 counties, observed 2010-2017 (4 years pre-ban, 4 years post-ban)
- AP-intensity varies across counties (mean ~1.5, cross-sectional)
- The AP ban in April 2013 sharply reduces "illegal immigrant" usage in high-AP-intensity outlets
- First-stage F = 23.63
- Reduced form: 1 SD higher AP-intensity leads to 0.7 pp lower support for border security after the ban
- 2SLS: 1 pp more "illegal immigrant" articles raises support by 0.5 pp

In [ ]:
np.random.seed(2023)

# --- Simulate county-level panel data ---
n_counties = 3100
years = np.arange(2010, 2018)
n_years = len(years)
ban_year = 2013

# County-level AP-intensity (time-invariant)
ap_intensity = np.random.exponential(1.5, n_counties)

# Build panel
county_ids = np.repeat(np.arange(n_counties), n_years)
year_vals = np.tile(years, n_counties)
ap_vals = np.repeat(ap_intensity, n_years)
post_ban = (year_vals >= ban_year).astype(float)

# Unobserved county-level conservatism (the confounder)
# Affects BOTH language choice AND immigration attitudes
conservatism = np.random.normal(0, 1, n_counties)
cons_vals = np.repeat(conservatism, n_years)

# --- Endogenous variable: "illegal immigrant" share ---
# Three components:
# 1. AP ban effect (exogenous, through the instrument)
# 2. County conservatism (endogenous, the confounder)
# 3. Noise
illimm_share = (0.40
                - 0.05 * ap_vals * post_ban   # instrument channel (exogenous)
                + 0.03 * cons_vals             # confounder channel (endogenous)
                + np.random.normal(0, 0.10, len(county_ids)))
illimm_share = np.clip(illimm_share, 0, 1)

# --- Outcome: support for border security ---
# True causal effect of illimm_share = 0.15 (per unit share, i.e. 0-1 scale)
true_causal = 0.15

support = (0.40
           + true_causal * illimm_share       # true causal effect of language
           + 0.04 * cons_vals                  # direct confounding effect
           + np.random.normal(0, 0.03, len(county_ids)))
support = np.clip(support, 0, 1)

df_dj = pd.DataFrame({
    'county': county_ids, 'year': year_vals,
    'ap_intensity': ap_vals, 'post_ban': post_ban,
    'illimm_share': illimm_share, 'support_border': support
})
df_dj['instrument'] = df_dj['ap_intensity'] * df_dj['post_ban']

print(f"Panel: {n_counties} counties x {n_years} years = {len(df_dj)} obs")
print(f"Pre-ban 'illegal immigrant' share: {df_dj[df_dj.post_ban==0]['illimm_share'].mean():.3f}")
print(f"Post-ban 'illegal immigrant' share: {df_dj[df_dj.post_ban==1]['illimm_share'].mean():.3f}")
print(f"Support for border security: mean={support.mean():.3f}")
print(f"\nTrue causal effect of 'illegal immigrant' share on support: {true_causal}")

In [ ]:
# --- Visualize the first stage: event study ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: "illegal immigrant" share by AP-intensity group over time
median_ap = np.median(ap_intensity)
df_dj['high_ap'] = df_dj['ap_intensity'] > median_ap

for label, mask, color in [('High AP-intensity', df_dj['high_ap'], 'darkorange'),
                             ('Low AP-intensity', ~df_dj['high_ap'], 'steelblue')]:
    means = df_dj[mask].groupby('year')['illimm_share'].mean()
    axes[0].plot(means.index, means.values, 'o-', color=color, label=label, markersize=6)

axes[0].axvline(x=2013, color='red', linestyle='--', alpha=0.7, label='AP ban (April 2013)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('"Illegal immigrant" share')
axes[0].set_title('First Stage: Language Usage Drops After AP Ban')
axes[0].legend()

# Right: support for border security by AP-intensity group over time
for label, mask, color in [('High AP-intensity', df_dj['high_ap'], 'darkorange'),
                             ('Low AP-intensity', ~df_dj['high_ap'], 'steelblue')]:
    means = df_dj[mask].groupby('year')['support_border'].mean()
    axes[1].plot(means.index, means.values, 'o-', color=color, label=label, markersize=6)

axes[1].axvline(x=2013, color='red', linestyle='--', alpha=0.7, label='AP ban (April 2013)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Support for border security')
axes[1].set_title('Reduced Form: Attitudes Shift After AP Ban')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Left: High-AP-intensity counties show a sharper decline in 'illegal immigrant'")
print("usage after the ban. This is the first stage.")
print("Right: Support for border security also drops more in high-AP-intensity counties")
print("after the ban. This is the reduced form.")

In [ ]:
# --- Run OLS vs 2SLS ---
# OLS: regress support on illimm_share (biased due to confounding)
ols_dj = smf.ols('support_border ~ illimm_share', data=df_dj).fit()

# First stage: regress illimm_share on instrument
fs_dj = smf.ols('illimm_share ~ instrument', data=df_dj).fit()

# Reduced form: regress support on instrument
rf_dj = smf.ols('support_border ~ instrument', data=df_dj).fit()

# Manual 2SLS
df_dj['illimm_hat'] = fs_dj.predict(df_dj)
iv_dj = smf.ols('support_border ~ illimm_hat', data=df_dj).fit()

# Wald: reduced form / first stage
wald_dj = rf_dj.params['instrument'] / fs_dj.params['instrument']

print("=== First Stage ===")
print(f"  Coef on AP-intensity x PostBan: {fs_dj.params['instrument']:.4f}")
print(f"  F-statistic: {fs_dj.fvalue:.1f}")
print()
print("=== Reduced Form ===")
print(f"  Coef on instrument: {rf_dj.params['instrument']:.5f}")
print()
print("=== Estimates ===")
print(f"  True causal effect:   {true_causal:.3f}")
print(f"  OLS estimate:         {ols_dj.params['illimm_share']:.3f}  (biased upward by confounding)")
print(f"  2SLS estimate:        {iv_dj.params['illimm_hat']:.3f}")
print(f"  Wald (RF/FS):         {wald_dj:.3f}")
print()
if abs(iv_dj.params['illimm_hat'] - true_causal) < abs(ols_dj.params['illimm_share'] - true_causal):
    print("IV is closer to the truth than OLS!")
else:
    print("Note: with finite samples and simulation noise, IV may not be exact.")
print(f"\nOLS picks up the confounding (conservatism affects both language and attitudes).")
print(f"IV uses only the variation in language coming from the AP ban, bypassing the confound.")

## 8. Real-Data Replication: Djourelova (2023)

Now we replicate the key results using **actual data** from the paper's ICPSR replication package (project 182482). The county-level dataset contains:

- **2,371 counties** observed 2007-2020
- **AP-intensity**: how much each county's local newspapers relied on Associated Press copy (pre-ban, time-invariant)
- **Instrument**: `APintI_post` = IHS(AP-intensity) $\times$ PostBan (post = year $\geq$ 2013)
- **First-stage outcome**: `num_IllimmImm` = share of articles using "illegal immigrant" among all "immigrant" articles
- **Downstream outcomes**: county-level Republican vote shares (presidential, House, Senate)

The IHS (inverse hyperbolic sine) transform is $\text{IHS}(x) = \log(x + \sqrt{x^2 + 1})$, similar to $\log$ but defined at zero.

In [ ]:
# Load the actual Djourelova (2023) replication data
# Source: ICPSR project 182482 (AEA Data and Code Repository, CC BY 4.0)
import os

DATA_URL = 'https://raw.githubusercontent.com/Persuasion-at-Scale/instrumental-variables/main/data/master_data_county.dta'
LOCAL_PATH = '../data/master_data_county.dta'

if os.path.exists(LOCAL_PATH):
    dj = pd.read_stata(LOCAL_PATH)
else:
    dj = pd.read_stata(DATA_URL)

print(f"Loaded: {dj.shape[0]:,} observations, {dj.shape[1]} variables")
print(f"Counties: {dj.fips.nunique():,}")
print(f"Years: {sorted(dj.year.dropna().astype(int).unique())}")
print(f"\nKey variables:")
print(f"  APintI      = IHS(AP-intensity), time-invariant:  mean={dj.APintI.mean():.2f}")
print(f"  APintI_post = APintI x PostBan, the INSTRUMENT:  mean={dj.APintI_post.mean():.2f}")
print(f"  num_IllimmImm = 'illegal immigrant' as % of 'immigrant': mean={dj.num_IllimmImm.dropna().mean():.2f}")
print(f"  repshare_pres = Republican presidential vote share: mean={dj.repshare_pres.dropna().mean():.3f}")

In [ ]:
# --- First Stage: Does the AP ban instrument predict media language? ---
# Panel regression: num_IllimmImm ~ APintI_post + county FE + year FE
# Clustered standard errors at the county level

from linearmodels.panel import PanelOLS

# Set up panel
dj_panel = dj.dropna(subset=['year', 'fips']).copy()
dj_panel['fips'] = dj_panel['fips'].astype(int)
dj_panel['year'] = dj_panel['year'].astype(int)
dj_panel = dj_panel.set_index(['fips', 'year'])

# First stage
fs_sample = dj_panel.dropna(subset=['num_IllimmImm', 'APintI_post'])
fs = PanelOLS(
    fs_sample['num_IllimmImm'],
    fs_sample[['APintI_post']],
    entity_effects=True, time_effects=True
).fit(cov_type='clustered', cluster_entity=True)

print("=" * 55)
print("FIRST STAGE: AP Ban Instrument -> Media Language")
print("=" * 55)
print(f"Dep var: 'illegal immigrant' as % of 'immigrant' articles")
print(f"Sample: {len(fs_sample):,} county-years, {fs_sample.index.get_level_values(0).nunique():,} counties\n")
print(f"  APintI x PostBan:  {fs.params['APintI_post']:.4f}")
print(f"  Std error:         {fs.std_errors['APintI_post']:.4f}")
print(f"  t-statistic:       {fs.tstats['APintI_post']:.2f}")
print(f"  First-stage F:     {fs.tstats['APintI_post']**2:.1f}")
print(f"\nInterpretation: After the AP ban, counties with 1 unit higher")
print(f"IHS(AP-intensity) see a {abs(fs.params['APintI_post']):.1f} pp DECLINE in")
print(f"'illegal immigrant' usage. F >> 10: strong instrument.")

In [ ]:
# --- Event Study: Visualize the first stage ---
# Split counties by median AP-intensity, plot "illegal immigrant" share over time

median_api = dj_panel['APintI'].median()
dj_panel['high_ap'] = dj_panel['APintI'] > median_api

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: First stage ("illegal immigrant" share by AP group)
for label, high, color in [('High AP-intensity', True, 'darkorange'),
                             ('Low AP-intensity', False, 'steelblue')]:
    sub = dj_panel[dj_panel['high_ap'] == high].dropna(subset=['num_IllimmImm'])
    means = sub.groupby(level='year')['num_IllimmImm'].mean()
    axes[0].plot(means.index, means.values, 'o-', color=color, label=label, markersize=5)

axes[0].axvline(x=2013, color='red', linestyle='--', alpha=0.7, label='AP ban (April 2013)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('"Illegal immigrant" as % of "immigrant"')
axes[0].set_title('First Stage: Language Change After AP Ban')
axes[0].legend(fontsize=9)

# Right: Republican vote share by AP group (election years only)
for label, high, color in [('High AP-intensity', True, 'darkorange'),
                             ('Low AP-intensity', False, 'steelblue')]:
    sub = dj_panel[dj_panel['high_ap'] == high].dropna(subset=['repshare_pres'])
    means = sub.groupby(level='year')['repshare_pres'].mean()
    axes[1].plot(means.index, means.values, 'o-', color=color, label=label, markersize=5)

axes[1].axvline(x=2013, color='red', linestyle='--', alpha=0.7, label='AP ban (April 2013)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Republican presidential vote share')
axes[1].set_title('Reduced Form: Vote Shares and AP Exposure')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Left: High-AP-intensity counties see a SHARPER decline in 'illegal")
print("immigrant' usage after 2013. The gap widens post-ban: that's the first stage.")
print("\nRight: Republican vote shares by AP-intensity group. Any differential")
print("shift after 2013 is the reduced form (instrument -> outcome).")

In [ ]:
# --- OLS vs IV: The punchline ---
# OLS: regress vote share on media language (biased by confounding)
# IV: instrument media language with AP ban instrument

# OLS (biased)
ols_sample = dj_panel.dropna(subset=['repshare_pres', 'num_IllimmImm'])
ols_real = PanelOLS(
    ols_sample['repshare_pres'],
    ols_sample[['num_IllimmImm']],
    entity_effects=True, time_effects=True
).fit(cov_type='clustered', cluster_entity=True)

# Reduced form
rf_sample = dj_panel.dropna(subset=['repshare_pres', 'APintI_post'])
rf_real = PanelOLS(
    rf_sample['repshare_pres'],
    rf_sample[['APintI_post']],
    entity_effects=True, time_effects=True
).fit(cov_type='clustered', cluster_entity=True)

# IV = reduced form / first stage (Wald estimator on same sample)
iv_sample = dj_panel.dropna(subset=['repshare_pres', 'num_IllimmImm', 'APintI_post']).copy()

# Demean for manual IV
for col in ['repshare_pres', 'num_IllimmImm', 'APintI_post']:
    em = iv_sample.groupby(level=0)[col].transform('mean')
    tm = iv_sample.groupby(level=1)[col].transform('mean')
    gm = iv_sample[col].mean()
    iv_sample[f'{col}_dm'] = iv_sample[col] - em - tm + gm

z = iv_sample['APintI_post_dm'].values
d = iv_sample['num_IllimmImm_dm'].values
y = iv_sample['repshare_pres_dm'].values

fs_coef = np.dot(z, d) / np.dot(z, z)
rf_coef = np.dot(z, y) / np.dot(z, z)
iv_coef = rf_coef / fs_coef

print("=" * 60)
print("COMPARISON: OLS vs IV (Real Data)")
print("=" * 60)
print(f"\n{'Method':<25} {'Coefficient':>12} {'Interpretation'}")
print("-" * 70)
print(f"{'OLS (biased)':<25} {ols_real.params['num_IllimmImm']:>12.6f}  More 'illegal immigrant' -> higher R vote share")
print(f"{'IV/2SLS (causal)':<25} {iv_coef:>12.6f}  IV REVERSES the sign!")
print(f"{'First stage':<25} {fs.params['APintI_post']:>12.4f}  AP ban reduces 'illegal immigrant' usage")
print(f"{'Reduced form':<25} {rf_real.params['APintI_post']:>12.6f}  AP ban -> slightly higher R vote share")
print()
print("OLS is POSITIVE: counties where newspapers use more 'illegal immigrant'")
print("also vote more Republican. But this reflects confounding (conservative")
print("counties have conservative newspapers AND vote Republican).")
print()
print("The IV estimate uses only the variation in language coming from the")
print("AP ban, which is exogenous to county ideology.")